# End-to-End ML Pipeline for Tabular Data 

In [1]:
# Data manipulation libraries
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder,PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.base import BaseEstimator, TransformerMixin

# Feature selection
from sklearn.feature_selection import mutual_info_regression, SelectKBest

# For transforming skewed features
from scipy import stats


# Configurations
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_style('whitegrid')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
# Set random seed for reproducibility
np.random.seed(42)

In [2]:
# Load the dataset
df = pd.read_csv('house.csv')

# Quick inspection
print(f"Dataset shape: {df.shape}")
print(f"Number of features: {df.shape[1] - 1}")  # Excluding SalePrice
df.head()

Dataset shape: (1460, 81)
Number of features: 80


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [4]:
# Separate features and target
X = df.drop(['Id', 'SalePrice'], axis=1)
y = df['SalePrice']

# Create train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (1022, 79)
Testing set shape: (438, 79)


### -------------------------
### 1) CategoricalMissingValueImputer
### -------------------------

In [5]:
class CategoricalMissingValueImputer(BaseEstimator, TransformerMixin):

    def __init__(self):
        # These columns always mean "No Feature"
        self.no_feature_cols = [
            'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
            'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish',
            'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature'
        ]
        self.modes_ = {}

    def fit(self, X, y=None):
        X = X.copy()
        categorical_cols = X.select_dtypes(include='object').columns

        # For regular categorical columns → compute mode
        for col in categorical_cols:
            if col not in self.no_feature_cols:
                self.modes_[col] = X[col].mode(dropna=True)[0]

        return self

    def transform(self, X):
        X = X.copy()
        categorical_cols = X.select_dtypes(include='object').columns

        # Fill "no feature" columns
        for col in categorical_cols:
            if col in self.no_feature_cols:
                X[col] = X[col].fillna('None')

        # Fill mode for remaining categorical columns
        for col in categorical_cols:
            if col not in self.no_feature_cols:
                if col in self.modes_:
                    X[col] = X[col].fillna(self.modes_[col])

        return X

### -------------------------
### 2) NumericalMissingValueImputer (with LotFrontage by Neighborhood)
### -------------------------

In [6]:
class NumericalMissingValueImputer(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.zero_fill_cols = [
            'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
            'TotalBsmtSF', 'GarageArea', 'GarageCars'
        ]
        self.medians_ = {}
        self.lotfrontage_by_neigh_ = {}
        self.lotfrontage_global_median_ = None

    def fit(self, X, y=None):
        X = X.copy()

        # Numeric columns (use a generic numeric selection)
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

        # Compute medians for numeric columns (excluding special-case cols)
        for col in numeric_cols:
            if col in self.zero_fill_cols or col == 'GarageYrBlt':
                # skip; handled specially in transform
                continue
            if col == 'LotFrontage':
                # handled separately below
                continue
            # store median (may be NaN if column all-null)
            self.medians_[col] = X[col].median()

        # LotFrontage: compute per-neighborhood medians (training data) and global median
        if 'LotFrontage' in X.columns:
            if 'Neighborhood' in X.columns:
                # groupby median will produce NaN for neighborhoods with all-NaN LotFrontage
                neigh_medians = X.groupby('Neighborhood')['LotFrontage'].median()
                # convert to dict (neighborhood -> median or NaN)
                self.lotfrontage_by_neigh_ = neigh_medians.to_dict()
            else:
                # no Neighborhood column — leave dict empty
                self.lotfrontage_by_neigh_ = {}

            # global median fallback (computed from training data LotFrontage)
            self.lotfrontage_global_median_ = X['LotFrontage'].median()

        return self

    def transform(self, X):
        X = X.copy()
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

        # A) Zero-fill
        for col in numeric_cols:
            if col in self.zero_fill_cols:
                X[col] = X[col].fillna(0)

        # B) GarageYrBlt → fill with YearBuilt if available
        if 'GarageYrBlt' in X.columns and 'YearBuilt' in X.columns:
            X['GarageYrBlt'] = X['GarageYrBlt'].fillna(X['YearBuilt'])

        # C) LotFrontage → neighborhood median (train) then global median fallback
        if 'LotFrontage' in X.columns:
            # If we computed neighborhood medians during fit and Neighborhood exists in X:
            if self.lotfrontage_by_neigh_ and 'Neighborhood' in X.columns:
                # Map neighborhoods to the training median (may produce NaN if neighborhood unseen or median NaN)
                mapped = X['Neighborhood'].map(self.lotfrontage_by_neigh_)

                # If mapped value is NaN (either unseen neighborhood or train median was NaN),
                # fallback to the training global median.
                fallback = self.lotfrontage_global_median_
                X['LotFrontage'] = X['LotFrontage'].fillna(mapped).fillna(fallback)
            else:
                # No neighborhood medians available; use global median if learned
                fallback = self.lotfrontage_global_median_ if self.lotfrontage_global_median_ is not None else 0
                X['LotFrontage'] = X['LotFrontage'].fillna(fallback)

        # D) Remaining numeric -> medians learned in fit()
        for col in numeric_cols:
            if col in self.zero_fill_cols or col == 'LotFrontage' or col == 'GarageYrBlt':
                continue
            if col in self.medians_:
                median_val = self.medians_[col]
                # if median is NaN (all-null during fit), skip or fill with 0 — we choose to skip to let caller decide
                if pd.notna(median_val):
                    X[col] = X[col].fillna(median_val)

        return X

### -------------------------
### 3) FeatureEngineer (creates features + drops helper columns)
### -------------------------

In [7]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        # Columns to drop after feature engineering
        self.cols_to_drop = [
            'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',  # Replaced by TotalSF
            'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath', # Replaced by TotalBathrooms
            'YearBuilt', 'YearRemodAdd', # Replaced by Age features
            'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', # Replaced by TotalPorchSF
            'PoolArea' # Replaced by HasPool
            # GarageArea, Fireplaces, OverallQual → DO NOT DROP
        ]

    def fit(self, X, y=None):
        # No fitting required; all operations are deterministic
        return self

    def transform(self, X):
        X = X.copy()

        # --- 1. Total Square Footage ---
        if all(c in X.columns for c in ['TotalBsmtSF', '1stFlrSF', '2ndFlrSF']):
            X['TotalSF'] = X['TotalBsmtSF'] + X['1stFlrSF'] + X['2ndFlrSF']

        # --- 2. Total Bathrooms ---
        if all(c in X.columns for c in ['FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath']):
            X['TotalBathrooms'] = (X['FullBath'] + 0.5 * X['HalfBath'] +
                                   X['BsmtFullBath'] + 0.5 * X['BsmtHalfBath'])

        # --- 3. House Age ---
        if all(c in X.columns for c in ['YrSold', 'YearBuilt']):
            X['HouseAge'] = X['YrSold'] - X['YearBuilt']

        # --- 4. Years Since Remodel ---
        if all(c in X.columns for c in ['YrSold', 'YearRemodAdd']):
            X['RemodAge'] = X['YrSold'] - X['YearRemodAdd']

        # --- 5. Has Pool ---
        if 'PoolArea' in X.columns:
            X['HasPool'] = (X['PoolArea'] > 0).astype(int)

        # --- 6. Has Garage ---
        if 'GarageArea' in X.columns:
            X['HasGarage'] = (X['GarageArea'] > 0).astype(int)

        # --- 7. Has Fireplace ---
        if 'Fireplaces' in X.columns:
            X['HasFireplace'] = (X['Fireplaces'] > 0).astype(int)

        # --- 8. Total Porch Area ---
        porch_cols = ['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']
        if all(c in X.columns for c in porch_cols):
            X['TotalPorchSF'] = X[porch_cols].sum(axis=1)

        # --- 9. Remodeled (binary) ---
        if all(c in X.columns for c in ['YearBuilt', 'YearRemodAdd']):
            X['Remodeled'] = (X['YearBuilt'] != X['YearRemodAdd']).astype(int)

        # --- 10. TotalQual ---
        if all(c in X.columns for c in ['OverallQual', 'OverallCond']):
            X['TotalQual'] = X['OverallQual'] * X['OverallCond']

        # --- Drop helper columns ---
        cols_to_drop_existing = [c for c in self.cols_to_drop if c in X.columns]
        if cols_to_drop_existing:
            X = X.drop(columns=cols_to_drop_existing)

        return X


### -------------------------
### 4) OrdinalEncoderTransformer (self-contained mappings)
### -------------------------

In [8]:
class OrdinalEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        # Hardcoded mappings inside the class
        quality_mapping = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
        basement_exposure_mapping = {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
        basement_finish_mapping = {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
        fence_mapping = {'None': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4}
        garage_finish_mapping = {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}
        lot_shape_mapping = {'Reg': 3, 'IR1': 2, 'IR2': 1, 'IR3': 0}
        functional_mapping = {'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3, 'Mod': 4, 'Min2': 5, 'Min1': 6, 'Typ': 7}

        self.ordinal_features = {
            'ExterQual': quality_mapping,
            'ExterCond': quality_mapping,
            'BsmtQual': quality_mapping,
            'BsmtCond': quality_mapping,
            'BsmtExposure': basement_exposure_mapping,
            'BsmtFinType1': basement_finish_mapping,
            'BsmtFinType2': basement_finish_mapping,
            'HeatingQC': quality_mapping,
            'KitchenQual': quality_mapping,
            'FireplaceQu': quality_mapping,
            'GarageQual': quality_mapping,
            'GarageCond': quality_mapping,
            'GarageFinish': garage_finish_mapping,
            'PoolQC': quality_mapping,
            'Fence': fence_mapping,
            'LotShape': lot_shape_mapping,
            'Functional': functional_mapping,
            'PavedDrive': {'N': 0, 'P': 1, 'Y': 2},
            'CentralAir': {'N': 0, 'Y': 1}
        }

    def fit(self, X, y=None):
        # Nothing to fit; mappings are predefined
        return self

    def transform(self, X):
        X = X.copy()
        for feature, mapping in self.ordinal_features.items():
            if feature in X.columns:
                # Map values; unseen/missing → 0
                X[feature] = X[feature].map(mapping).fillna(0).astype(int)
        return X

### -------------------------
### 5) OneHotEncoderTransformer (self-contained nominal list)
### -------------------------

In [9]:
class OneHotEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        # Hardcoded nominal features
        self.nominal_features = [
            'MSZoning', 'Street', 'Alley', 'LandContour', 'LotConfig', 'Neighborhood',
            'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl',
            'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'Electrical',
            'GarageType', 'MiscFeature', 'SaleType', 'SaleCondition','Utilities', 'LandSlope'
        ]
        self.categories_ = {}  # Will store training categories for each feature

    def fit(self, X, y=None):
        X = X.copy()
        # Keep only nominal features present in dataset
        self.nominal_features = [f for f in self.nominal_features if f in X.columns]
        for feature in self.nominal_features:
            # Store unique categories in training set
            self.categories_[feature] = X[feature].dropna().unique().tolist()
        return self

    def transform(self, X):
        X = X.copy()
        for feature in self.nominal_features:
            if feature not in X.columns:
                continue
            categories = self.categories_.get(feature, [])
            # Create dummy columns, drop first to avoid multicollinearity
            for cat in categories[1:]:
                col_name = f"{feature}_{cat}"
                X[col_name] = (X[feature] == cat).astype(int)
            # Drop original column
            X = X.drop(columns=feature)
        return X

### -------------------------
### 6) NumericalTransformer (PowerTransformer on skewed features + StandardScaler)
### -------------------------

In [10]:
class NumericalTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, skew_threshold=0.75):
        self.skew_threshold = skew_threshold
        self.skewed_features = []
        self.numeric_columns = []
        self.power_transformer = None
        self.scaler = None

    def fit(self, X, y=None):
        X = X.copy()

        # Identify numeric columns
        self.numeric_columns = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

        # Identify skewed numeric features
        self.skewed_features = [
            col for col in self.numeric_columns if abs(X[col].skew()) > self.skew_threshold
        ]

        # Fit PowerTransformer ONLY on skewed features
        if self.skewed_features:
            self.power_transformer = PowerTransformer(method='yeo-johnson')
            self.power_transformer.fit(X[self.skewed_features])

        # Fit StandardScaler on ALL numeric columns
        self.scaler = StandardScaler()
        self.scaler.fit(X[self.numeric_columns])

        return self

    def transform(self, X):
        X = X.copy()

        # Apply power transformation to skewed features
        if self.skewed_features:
            X[self.skewed_features] = self.power_transformer.transform(
                X[self.skewed_features]
            )

        # Apply scaling to all numeric features
        X[self.numeric_columns] = self.scaler.transform(X[self.numeric_columns])

        # Always return a DataFrame, not a NumPy array
        return X

### -------------------------
### 7) MutualInformationSelector
### -------------------------

In [11]:
class MutualInformationSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.1, log_target=True):
        self.threshold = threshold
        self.log_target = log_target
        self.selected_features_ = None
        self.mi_scores_ = None

    def fit(self, X, y=None):
        X = X.copy()

        # Apply log1p transformation to target if enabled
        if self.log_target:
            y_transformed = np.log1p(y)
        else:
            y_transformed = y

        # Compute mutual information scores
        mi = mutual_info_regression(X, y_transformed)
        mi_scores = pd.Series(mi, index=X.columns)

        # Sort MI scores
        self.mi_scores_ = mi_scores.sort_values(ascending=False)

        # Select features above threshold
        self.selected_features_ = self.mi_scores_[self.mi_scores_ > self.threshold].index.tolist()

        return self

    def transform(self, X):
        X = X.copy()

        # Reduce to selected features only
        return X[self.selected_features_]


### -------------------------
### Build the assembled pipeline
### -------------------------

In [12]:
# -------------------------
# Build the assembled pipeline
# -------------------------
def build_full_pipeline(model=None, mi_threshold=0.1):
    # Default model
    if model is None:
        model = LinearRegression()

    pipeline = Pipeline([
        ('cat_imputer', CategoricalMissingValueImputer()),
        ('num_imputer', NumericalMissingValueImputer()),
        ('ordinal_encoder', OrdinalEncoderTransformer()),
        ('onehot_encoder', OneHotEncoderTransformer()),
        ('feature_engineer', FeatureEngineer()),
        ('num_transform', NumericalTransformer(skew_threshold=0.75)),
        # MutualInformationSelector expects y already transformed if log_target=False.
        ('feature_selector', MutualInformationSelector(threshold=mi_threshold, log_target=False)),
        ('model', model)
    ])
    return pipeline

### -------------------------
### Utility: evaluation function
### -------------------------

In [13]:
# -------------------------
# Utility: evaluation function
# -------------------------
def evaluate_model(pipeline, X_test, y_test):
    # pipeline predicts log prices (we will fit with log1p(y_train))
    y_pred_log = pipeline.predict(X_test)
    y_pred_dollars = np.expm1(y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(np.log1p(y_test), y_pred_log))
    rmse_dollars = np.sqrt(mean_squared_error(y_test, y_pred_dollars))
    r2 = r2_score(y_test, y_pred_dollars)
    return {"RMSE (log scale)": rmse_log, "RMSE (dollars)": rmse_dollars, "R2 (dollars)": r2}


###############################################################################
### MODEL PIPELINES (LR, SVR, Decision Tree)
###############################################################################

In [14]:
###############################################################################
# 2. MODEL PIPELINES (LR, SVR, Decision Tree)
###############################################################################

# FULL PIPELINE → For Linear Regression
linear_reg_pipeline = build_full_pipeline(
    model=LinearRegression()
)
linear_reg_pipeline.fit(X_train, np.log1p(y_train))


# FULL PIPELINE → For SVR
svr_pipeline = build_full_pipeline(
    model=SVR(kernel='rbf', C=1.0, epsilon=0.1)
)
svr_pipeline.fit(X_train, np.log1p(y_train))


# FULL PIPELINE → For Decision Tree 
decision_tree_pipeline = build_full_pipeline(
    model=DecisionTreeRegressor(max_depth = 10, random_state=42)
)
decision_tree_pipeline.fit(X_train, np.log1p(y_train))

Pipeline(steps=[('cat_imputer', CategoricalMissingValueImputer()),
                ('num_imputer', NumericalMissingValueImputer()),
                ('ordinal_encoder', OrdinalEncoderTransformer()),
                ('onehot_encoder', OneHotEncoderTransformer()),
                ('feature_engineer', FeatureEngineer()),
                ('num_transform', NumericalTransformer()),
                ('feature_selector',
                 MutualInformationSelector(log_target=False)),
                ('model',
                 DecisionTreeRegressor(max_depth=10, random_state=42))])

###############################################################################
### EVALUATE ALL MODELS
###############################################################################

In [16]:
###############################################################################
# 4. FIT & EVALUATE ALL MODELS
###############################################################################

results_lr = evaluate_model(
    linear_reg_pipeline,  X_test, y_test
)
print(results_lr)


results_svr = evaluate_model(
    svr_pipeline, X_test, y_test
)
print(results_svr)


results_dt = evaluate_model(
    decision_tree_pipeline,  X_test,  y_test
)
print(results_dt)

{'RMSE (log scale)': 0.13464491721190536, 'RMSE (dollars)': 27633.56177949535, 'R2 (dollars)': 0.8905698658141278}
{'RMSE (log scale)': 0.15284251540039404, 'RMSE (dollars)': 31519.543477392006, 'R2 (dollars)': 0.8576285078620849}
{'RMSE (log scale)': 0.1919912649952187, 'RMSE (dollars)': 35303.73351640137, 'R2 (dollars)': 0.8213905340203809}


In [ ]:
Updating the Piple

In [17]:
###############################################################################
# 1. PIPELINE BUILDER
###############################################################################

def build_full_pipeline(
    model=None,
    mi_threshold=0.1,
    apply_cat_imputer=True,
    apply_num_imputer=True,
    apply_ordinal=True,
    apply_onehot=True,
    apply_feature_engineering=True,
    apply_num_transform=True,
    apply_feature_selection=True
):

    if model is None:
        model = LinearRegression()

    steps = []

    if apply_cat_imputer:
        steps.append(('cat_imputer', CategoricalMissingValueImputer()))

    if apply_num_imputer:
        steps.append(('num_imputer', NumericalMissingValueImputer()))

    if apply_ordinal:
        steps.append(('ordinal_encoder', OrdinalEncoderTransformer()))

    if apply_onehot:
        steps.append(('onehot_encoder', OneHotEncoderTransformer()))

    if apply_feature_engineering:
        steps.append(('feature_engineer', FeatureEngineer()))

    if apply_num_transform:
        steps.append(('num_transform', NumericalTransformer(skew_threshold=0.75)))

    if apply_feature_selection:
        steps.append(('feature_selector', MutualInformationSelector(
            threshold=mi_threshold,
            log_target=False
        )))

    steps.append(('model', model))
    return Pipeline(steps)

In [18]:
###############################################################################
# 3. FUNCTION TO TRAIN, PREDICT AND EVALUATE MODELS
###############################################################################

def evaluate_pipeline(pipeline, X_train, X_test, y_train, y_test, model_name="Model"):

    # Log-transform target
    y_train_log = np.log1p(y_train)
    y_test_log = np.log1p(y_test)

    # Train
    pipeline.fit(X_train, y_train_log)

    # Predict log-values
    y_pred_log = pipeline.predict(X_test)

    # RMSE on log scale
    rmse_log = np.sqrt(mean_squared_error(y_test_log, y_pred_log))

    # Convert predictions to dollars
    y_pred_dollars = np.expm1(y_pred_log)

    # R² on dollar values
    r2 = r2_score(y_test, y_pred_dollars)

    # RMSE in dollars
    rmse_dollars = np.sqrt(mean_squared_error(y_test, y_pred_dollars))

    print(f"\n============================")
    print(f" RESULTS: {model_name}")
    print(f"============================")
    print(f"RMSE (log scale):       {rmse_log:.4f}")
    print(f"R² (dollars):           {r2:.4f}")
    print(f"RMSE (dollars):         {rmse_dollars:.4f}")

    return {
        "model": model_name,
        "rmse_log": rmse_log,
        "rmse_dollars": rmse_dollars,
        "r2": r2,
        "pipeline": pipeline
    }

In [19]:
###############################################################################
# 2. MODEL PIPELINES (LR, SVR, Decision Tree)
###############################################################################

# FULL PIPELINE → For Linear Regression
linear_reg_pipeline = build_full_pipeline(
    model=LinearRegression()
)

# FULL PIPELINE → For SVR
svr_pipeline = build_full_pipeline(
    model=SVR(kernel='rbf', C=1.0, epsilon=0.1)
)

# MODIFIED PIPELINE → For Decision Tree (NO NUMERICAL TRANSFORMATION)
decision_tree_pipeline = build_full_pipeline(
    model=DecisionTreeRegressor(max_depth = 10, random_state=42),
    apply_num_transform=False   # <--- key difference
)

In [20]:
###############################################################################
# 4. FIT & EVALUATE ALL MODELS
###############################################################################

results_lr = evaluate_pipeline(
    linear_reg_pipeline, X_train, X_test, y_train, y_test, model_name="Linear Regression"
)

results_svr = evaluate_pipeline(
    svr_pipeline, X_train, X_test, y_train, y_test, model_name="SVR"
)

results_dt = evaluate_pipeline(
    decision_tree_pipeline, X_train, X_test, y_train, y_test, model_name="Decision Tree"
)


 RESULTS: Linear Regression
RMSE (log scale):       0.1340
R² (dollars):           0.8901
RMSE (dollars):         27689.9448

 RESULTS: SVR
RMSE (log scale):       0.1529
R² (dollars):           0.8577
RMSE (dollars):         31511.2049

 RESULTS: Decision Tree
RMSE (log scale):       0.1933
R² (dollars):           0.8333
RMSE (dollars):         34111.1590
